# 02 — Y2Y corridor-wide solve (Gate-0 arms)

Runs prioritizr on the **full** aligned stack (no crop) — the corridor-wide solve, over the shared
`prioritizr_core.R` engine. Parameters come from `config.ANALYSES["y2y"]`, carried via
`manifest.json` (cell 1 refreshes it).

**Configure a run: edit cell 2, the RUN LEVER.** It carries the **Gate-0 arms** of the
frequency-ensemble study (`analyses/y2y/spec/`): `a0_control` / `a1_protocol` / `a2_flat30` /
`a3_flat40` (+ optional `a4_pullcheck`), each a `(TARGETS, WEIGHTS)` pair with **w = t** so only
the stopping point varies between arms. Uncomment one block, run top-to-bottom, repeat. The arm
dicts come from the frozen Gate-0a table — run `01_feature_audit.ipynb` first;
verdicts come after — run `03_gate0_validation.ipynb` when all arms are solved.

Overrides are applied to the run context, not written back into `manifest.json`;
`run_summary.json` records the parameters **actually solved**, so that is the file to compare
runs on. `pr_override` refuses to overwrite a folder whose recorded targets *or weights* differ.

**Kernel:** `R (y2y)`. Run cell-by-cell; Ethan runs, Claude never executes.

In [1]:
# ---- Setup: shared engine + this analysis' key + manifest refresh --------
# (source() moved below the bootstrap so the path resolves from any cwd)
ANALYSIS <- "y2y"            # <-- 03b/03c (still at repo root) differ only in this line
# Root-finding bootstrap: this notebook lives in analyses/y2y/, below the repo root where
# config.py and prioritizr_core.R sit. Same pattern as run_one.R.
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))   # pr_* functions

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # regenerate manifest.json from config for THIS
ctx   <- pr_setup(mpath, PROJ)                 # analysis (stops on failure); print the banner

manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y


In [ ]:
# ============ THE RUN LEVER — Gate-0 arms: edit THIS cell, run the notebook ============
# One arm per pass: uncomment exactly ONE block, run top-to-bottom, repeat. Everything else comes
# from config.py. The arm dicts are DERIVED by 01_feature_audit.ipynb (this folder) from the frozen
# Gate-0a table -- run that notebook first; its last cell prints these blocks, and this cell must
# agree with it. Verdicts afterwards: 03_gate0_validation.ipynb (this folder).
#
# A target is a STOPPING RULE. Under min-shortfall the objective depends on w/t below target
# ("pull") and is flat above it. ALL ARMS SET WEIGHTS <- TARGETS (w = t), so pull stays 1.00 for
# every feature and the ONLY thing varying across arms is the stopping point -- at w = 1 a target
# of 0.332 raises carbon's pull to 3.0x and its share of objective swing to ~55%, confounding the
# comparison (that superseded configuration ran once as iter7_y2y_r1_density5x).
#
# MEASURED, from that run (2026-08-18): (1) m_soc parked at EXACTLY its 0.332 target; (2) biomass
# landed at 0.259 vs its 0.066 target -- min-shortfall never penalizes EXCEEDING a target, so a
# satiated feature free-rides on cells picked for other values; excess above target is expected
# co-capture, not a failure. (3) solve time 4,289 s (~71 min, almost all HiGHS presolve) -- NOT
# the ~12 s of the untargeted iter2 LP. Expect a0 fast and a1-a3 possibly ~1 h each.
#
# Do NOT add targets to the foundational features: if every target became simultaneously
# achievable the objective would go flat and the solver would return an arbitrary optimum.

# ---- the Gate-0 arms — uncomment exactly ONE ----------------------------------
# a0 control: no carbon change. Isolates penalty-removal + 1/v; brackets S4 (extreme carbon-forward).
RUN <- "a0_control";  TARGETS <- list()

# a1 the protocol configuration -- m_soc ONLY (biomass REVERTED to weight-levered by R2's
#    tail-mass criterion: implied target 0.066 < t_min 0.15; its ~40%+ capture here is expected).
# RUN <- "a1_protocol"; TARGETS <- list(irrecoverable_carbon_m_soc = 0.332)

# a2 flat 30%: both pools at exactly area share.
# RUN <- "a2_flat30";   TARGETS <- list(irrecoverable_carbon_m_soc   = 0.30,
#                                       irrecoverable_carbon_biomass = 0.30)

# a3 flat 40%: mild demotion; biomass binds barely (control captures 41.8%) -- watch it.
# RUN <- "a3_flat40";   TARGETS <- list(irrecoverable_carbon_m_soc   = 0.40,
#                                       irrecoverable_carbon_biomass = 0.40)

# a4 OPTIONAL pull-invariance check (gate G-uniform). t must be UNREACHABLE so the target can do
#    nothing (connectivity cap_max = 0.552 -> t = 0.6 is beyond any feasible capture); then
#    w = t = 0.6 leaves pull at 1.00 and the solution MUST reproduce a0 exactly -- the empirical
#    proof that only w/t and the stopping point matter.
# RUN <- "a4_pullcheck"; TARGETS <- list(transboundary_connectivity = 0.6)
# ------------------------------------------------------------------------------

WEIGHTS <- TARGETS    # w = t on every arm: pull 1.00; only the stopping point varies

# DIRECT assignment, deliberately NOT `modifyList(ctx, pr_override(...))`: modifyList deep-merges
# nested lists, so an arm's TARGETS would MERGE with config.py's baseline instead of replacing it
# (a0's empty list would clear nothing and the control would quietly solve the baseline target).
# pr_override returns the full updated ctx and prints the EFFECTIVE targets/weights -- on a0 that
# line must read "<none>"; check it before solving.
ctx <- pr_override(ctx,
    targets                    = TARGETS,
    feature_weight_multipliers = WEIGHTS,
    results_subdir             = paste0("iter7_y2y_", RUN))

In [3]:
# ---- Ingest the stack + crop to the ROI + normalize (window set in config.ANALYSES) ----
ctx <- modifyList(ctx, pr_ingest(ctx))

ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)


In [4]:
# ---- Planning units + lock-in + feasibility check ----
ctx <- modifyList(ctx, pr_planning_units(ctx))

planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget


In [5]:
# ---- Feature weights (+ any per-analysis up-weighting) ----
ctx <- modifyList(ctx, pr_weights(ctx))

weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)


In [6]:
# ---- Per-feature relative targets (from the RUN LEVER cell above) ----
# Applies the target vector: every feature at the default target_pct, then the overrides. Built
# BY NAME off names(features), so it cannot silently misalign if the stack order changes; an
# unknown feature name or an out-of-range value stops the run here rather than at the solve.
# Check the printed overrides match the lever cell before solving.
ctx <- modifyList(ctx, pr_targets(ctx))

targets: default 1.00; 2 override(s):
  irrecoverable_carbon_m_soc         0.332
  irrecoverable_carbon_biomass       0.066


In [7]:
# ---- Spatial-penalty matrices (built only for penalties > 0) ----
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))

penalties -> connectivity=0 | boundary=0 | neighbor=0  (0 = off)


In [8]:
# ---- Build the conservation problem ----
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.066 and 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      highs solver (`gap` = 0.1, `time_limit` = 43200, …)
# ℹ Use `summary(...)` to see complete formulation.



In [9]:
# ---- Solve (Ethan runs; heavy) -- HiGHS single solution / Gurobi portfolio ----
# A run that hits the time limit returns an INFEASIBLE point (area > budget) -- discard it.
sv <- pr_solve(ctx); ctx$s <- sv$s; ctx$timing <- sv$timing; ctx$n_sol <- sv$n_sol

LP has 49 rows; 1272962 cols; 16976357 nonzeros

Coefficient ranges:

  Matrix  [1e-06, 1e+05]

  Cost    [3e-02, 1e+00]

  Bound   [1e+00, 1e+00]

  RHS     [7e+03, 4e+05]


Presolving model

48 rows, 1081932 cols, 13395827 nonzeros 3s

2 rows, 983658 cols, 1967073 nonzeros 4158s

Presolve reductions: rows 2(-47); columns 983658(-289304); nonzeros 1967073(-15009284) 

Solving the presolved LP

IPX model has 2 rows, 983658 columns and 1967073 nonzeros

Input
    Number of variables:                                983658
    Number of free variables:                           0
    Number of constraints:                              2
    Number of equality constraints:                     0
    Number of matrix entries:                           1967073

    Matrix range:                                       [1e-06, 3e+04]

    RHS range:                                          [2e+04, 2e+05]

    Objective range:                                    [2e-06, 1e+00]

    Bounds range:  

In [10]:
# ---- Per-alternative summaries + selection-frequency map ----
ctx <- modifyList(ctx, pr_summaries(ctx))

  alternative n_selected pct_region n_added_beyond_pa
1      alt_01   381874.2         30            190844


In [11]:
# ---- Write outputs for 04 (portfolio / frequency / representation / run_summary) ----
pr_write_outputs(ctx)

wrote:
  output_data/iter7_y2y_r1_density5x/portfolio.tif
  output_data/iter7_y2y_r1_density5x/selection_frequency.tif
  output_data/iter7_y2y_r1_density5x/portfolio_representation.csv
  output_data/iter7_y2y_r1_density5x/run_summary.json
